# Skills, instructions, and runtime scaffolding

A skill is not a prompt. It is a file a team maintains, reviews, versions and tests, and the text a model receives should be something the harness *builds* from it. This notebook makes that distinction concrete and then measures what it buys you.

**The four artifact types, and what each one is allowed to decide:**

| Artifact | Lifetime | Decides | Example rule |
|---|---|---|---|
| `AGENTS.md` — identity | durable, always loaded | what the agent is and never does | "never state a number that is not in the input" |
| `PROJECT.md` — project instructions | durable, always loaded | facts about this product | "a release is breaking when a caller must change code" |
| `persona-*.md` — persona | swappable per channel | how a sentence sounds | "short sentences, concrete nouns" |
| `skill-*.md` — skill | loaded per task | what the output *is* | "between three and six bullets, one change per bullet" |


**What this notebooks builds:** parse the artifacts into typed, hashed, versioned records → route a task to a skill and measure what a vague description costs → detect a boundary collision between two skills → compile the runtime surface for one task (identity + project + persona + only the slices of the skill that apply) → measure it against injecting everything → test the skill as a unit → version it and show what a change moves. The domain is release notes for a fictional data platform, with two axes: `tone` (formal, casual) and `audience` (customer, developer).


In [ ]:
!pip install instructor

## 0. The artifacts

Five files, four artifact types: one identity, one set of project instructions, one persona and two skills. They live in a shared Drive folder, they are plain Markdown with YAML frontmatter.

In [ ]:
try:
    import gdown
except ModuleNotFoundError:
    %pip install -q gdown
    import gdown

import hashlib, inspect, json, os, re, shutil, textwrap, time
from dataclasses import dataclass, field
from functools import lru_cache
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import yaml

DRIVE_FOLDER_URL = (
    "https://drive.google.com/drive/folders/"
    "1hPCzmDY08tKq5o8vcykap9aFdiQ7UA_4?usp=sharing"
)

WORK = Path("ch03_skill_demo").resolve()

# gdown preserves the Google Drive folder name.
SRC = WORK / "artifacts"

shutil.rmtree(WORK, ignore_errors=True)
WORK.mkdir(parents=True)

kwargs = {
    "url": DRIVE_FOLDER_URL,
    "output": str(WORK),
    "quiet": True,
    "use_cookies": False,
}

if "remaining_ok" in inspect.signature(gdown.download_folder).parameters:
    kwargs["remaining_ok"] = True

downloaded = gdown.download_folder(**kwargs)

if not downloaded:
    raise RuntimeError(
        "Nothing was downloaded. Ensure that the Google Drive folder is "
        "shared with 'Anyone with the link'."
    )

files = sorted(SRC.glob("*.md"))

if not files:
    raise RuntimeError(f"No Markdown artifacts found in {SRC}/")

print(f"{len(files)} artifact(s) in {SRC}:")
for p in files:
    print(f"   {p.name:28} {p.stat().st_size:>6} bytes")

5 artifact(s) in /content/ch03_skill_demo/artifacts:
   AGENTS.md                      1023 bytes
   PROJECT.md                      891 bytes
   persona-plainspoken.md          571 bytes
   skill-incident-report.md       1429 bytes
   skill-release-notes.md         3645 bytes


## 1. Artifacts as records

- **`owns`** lists the section ids this artifact is responsible for. It is a
  claim, and section 4 checks it against the other artifacts.
- **`always_loaded`** separates durable identity from loadable capability. It is
  the difference between *who the agent is* and *what it is doing right now*.
- **`when_to_use`** is the skill's interface to the selector. Section 3 shows how
  much rides on that one sentence.

Sections are delimited by `<!-- section: id -->`, and the id is what the rest of the system names. A rule that cannot be named cannot be tested, traced, or selectively loaded, which is why the id exists before any of the machinery does.


In [ ]:
_FRONTMATTER = re.compile(r"\A---\n(.*?)\n---\n?", re.S)
_SECTION = re.compile(r"^<!--\s*section:\s*([\w.\-]+)\s*-->\s*$", re.M)


@dataclass(frozen=True)
class Artifact:
    kind: str                       # identity | project | persona | skill
    name: str
    version: str
    path: Path
    meta: Dict
    sections: Dict[str, str]        # id -> body, in file order
    digest: str                     # content hash of the file as authored

    @property
    def always_loaded(self) -> bool:
        return bool(self.meta.get("always_loaded"))

    @property
    def owns(self) -> List[str]:
        return list(self.meta.get("owns") or [])

    @property
    def axes(self) -> List[str]:
        return list(self.meta.get("axes") or [])


def parse(path: Path) -> Artifact:
    raw = path.read_text(encoding="utf-8")
    m = _FRONTMATTER.match(raw)
    if not m:
        raise ValueError(f"{path.name}: no frontmatter — not an artifact")
    meta = yaml.safe_load(m.group(1)) or {}
    body = raw[m.end():]

    # Split on the section markers. A body outside any marker is a file someone
    # wrote as prose; it would be unaddressable, so it is an error, not a warning.
    marks = list(_SECTION.finditer(body))
    if not marks:
        raise ValueError(f"{path.name}: no <!-- section: ... --> markers")
    if body[:marks[0].start()].strip():
        raise ValueError(f"{path.name}: text before the first section marker")
    sections = {}
    for i, mark in enumerate(marks):
        end = marks[i + 1].start() if i + 1 < len(marks) else len(body)
        sections[mark.group(1)] = body[mark.end():end].strip()

    for missing in set(meta.get("owns") or []) - set(sections):
        raise ValueError(f"{path.name}: claims to own '{missing}' but has no such section")

    return Artifact(kind=meta["artifact"], name=meta["name"], version=str(meta["version"]),
                    path=path, meta=meta, sections=sections,
                    digest=hashlib.sha256(raw.encode()).hexdigest()[:12])


ARTIFACTS = {a.name: a for a in (parse(p) for p in files)}

print(f"{'artifact':22}{'kind':10}{'version':10}{'digest':14}{'always':8}sections")
for a in ARTIFACTS.values():
    print(f"{a.name:22}{a.kind:10}{a.version:10}{a.digest:14}"
          f"{'yes' if a.always_loaded else 'no':8}{', '.join(a.sections)}")


artifact              kind      version   digest        always  sections
acme-docs-agent       identity  2.1.0     f13e99c92930  yes     identity.role, identity.refusals, identity.escalation
acme-data-platform    project   4.0.2     6246496911d7  yes     project.product, project.versioning, project.audiences
plainspoken           persona   1.2.0     dc78ba66e414  no      persona.voice
incident-report       skill     1.1.0     a998ee689b8b  no      output.contract, summary.style, timeline.structure, prohibitions.hard, typography
release-notes         skill     3.4.0     9a0b8bce844f  no      output.contract, summary.style, highlights.structure, address.contract, prohibitions.hard, typography, examples


## 2. Durable identity, loadable capability

The **always-loaded** part is what the agent is when it is doing nothing in
particular: its refusals, the product's facts. Those hold across every task, and a skill is not allowed to weaken them. The **loadable** part is a capability, and it is *scoped*: it says what a release note is, and its rules stop at the edge of that task.

In [ ]:
ALWAYS = [a for a in ARTIFACTS.values() if a.always_loaded]
LOADABLE = [a for a in ARTIFACTS.values() if not a.always_loaded]

print("always in context (identity of the agent):")
for a in ALWAYS:
    print(f"   {a.kind:9} {a.name} {a.version}  ->  {', '.join(a.owns)}")
print("\nchosen per task (capability and voice):")
for a in LOADABLE:
    print(f"   {a.kind:9} {a.name} {a.version}  ->  {', '.join(a.sections)}")

print("\nthe refusals, which no skill may remove:")
print(textwrap.indent(ARTIFACTS["acme-docs-agent"].sections["identity.refusals"], "   "))


always in context (identity of the agent):
   identity  acme-docs-agent 2.1.0  ->  identity.role, identity.refusals, identity.escalation
   project   acme-data-platform 4.0.2  ->  project.product, project.versioning, project.audiences

chosen per task (capability and voice):
   persona   plainspoken 1.2.0  ->  persona.voice
   skill     incident-report 1.1.0  ->  output.contract, summary.style, timeline.structure, prohibitions.hard, typography
   skill     release-notes 3.4.0  ->  output.contract, summary.style, highlights.structure, address.contract, prohibitions.hard, typography, examples

the refusals, which no skill may remove:
   ## What you never do

   - Never state a number, date, version or endpoint that is not in the input.
   - Never promise a future release, a roadmap item or a deprecation date.
   - Never describe the impact on a named customer.

   These hold in every task, under every persona, in every skill. A skill may add
   prohibitions; it may not remove one of thes

## 3. Selecting a skill and why its description matters

When a harness has multiple skills available, it needs to select the appropriate one for the task. In this example, the selector uses the `when_to_use` descriptions to make that decision. The skill’s name provides a label, while its description defines when it applies. The full skill instructions are loaded only after selection.

The following experiment compares two tasks across two versions of the skill catalogue. The first task clearly calls for release notes: a version has shipped, and the merged changes are provided. The second task includes both an outage and a release, making the choice less straightforward.

You first use the original descriptions, then repeat the experiment with a less specific release-note description: “Use for writing customer-facing text about changes.” This gives you four combinations of task and catalogue.

For each combination, you run three samples at temperature 1. Repeated samples help us see whether the selector consistently chooses the same skill or whether its choice varies across runs. A single sample wouldn’t reveal that variation.

You examine both **the selected skill** and **the reason given for selecting it**. The model may still choose the appropriate skill with a vague description because it can infer the intended use from the task and the skill’s name. However, the vague description no longer provides an explicit selection criterion that we can inspect and test. A correct selection alone therefore doesn’t tell you whether the description provides enough information for reliable routing.

In [ ]:
import getpass, urllib.request

import instructor
import openai
from pydantic import BaseModel, Field

from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=False)

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key: ")

BASE_URL = "https://openrouter.ai/api/v1"
WRITER = "openai/gpt-5.6-luna"          # writes the notes
ROUTER = "openai/gpt-5.4-mini"          # picks the skill; a small model is plenty

USAGE: List[Dict[str, object]] = []      # every call, for the cost section at the end


@lru_cache(maxsize=4)
def client_for(mode):
    return instructor.from_openai(
        openai.OpenAI(api_key=os.environ["OPENROUTER_API_KEY"], base_url=BASE_URL),
        mode=mode)


def typed_call(*, model, response_model, messages, temperature=0.0, label="",
               mode=instructor.Mode.TOOLS, max_retries=2):
    """One structured call, with its usage and wall clock recorded."""
    t0 = time.perf_counter()
    out, completion = client_for(mode).chat.completions.create_with_completion(
        model=model, response_model=response_model, max_retries=max_retries,
        temperature=temperature, messages=messages)
    u = getattr(completion, "usage", None)
    USAGE.append({"label": label or model, "model": model,
                  "prompt_tokens": getattr(u, "prompt_tokens", 0),
                  "completion_tokens": getattr(u, "completion_tokens", 0),
                  "seconds": time.perf_counter() - t0})
    return out


print(f"writer: {WRITER}   router: {ROUTER}")


writer: openai/gpt-5.6-luna   router: openai/gpt-5.4-mini


In [ ]:
import collections


class SkillChoice(BaseModel):
    skill: str = Field(description="The name of the skill to load, exactly as listed.")
    why: str = Field(description="One short sentence.")


TASKS = {
    "unambiguous": ("Version 2.4.0 went out this morning. Here are the merged "
                    "changes; write the notes our customers will see."),
    # Both readings are defensible: an outage happened, and a version shipped.
    "ambiguous": ("Exports were failing for about an hour on Tuesday. We shipped "
                  "2.4.1 with the fix this morning — write the customer-facing text."),
}

VAGUE = "Use for writing customer-facing text about changes."
SAMPLES = 3


def catalogue(descriptions: Dict[str, str]) -> str:
    return "\n".join(f"- {name}: {desc.strip()}" for name, desc in descriptions.items())


def select(descriptions: Dict[str, str], task: str, label: str) -> SkillChoice:
    return typed_call(
        model=ROUTER, response_model=SkillChoice, label=label, temperature=1.0,
        messages=[{"role": "system", "content":
                   "Choose exactly one skill for the task. Answer with its name."},
                  {"role": "user", "content":
                   f"SKILLS\n{catalogue(descriptions)}\n\nTASK\n{task}"}])


AS_WRITTEN = {a.name: a.meta["when_to_use"] for a in LOADABLE if a.kind == "skill"}
DEGRADED = {**AS_WRITTEN, "release-notes": VAGUE}
CATALOGUES = {"as written": AS_WRITTEN, "degraded": DEGRADED}

picks, reasons = {}, {}
for task_name, task in TASKS.items():
    for cat_name, descriptions in CATALOGUES.items():
        votes = [select(descriptions, task, f"select[{task_name}/{cat_name}]")
                 for _ in range(SAMPLES)]
        picks[(task_name, cat_name)] = collections.Counter(v.skill for v in votes)
        reasons[(task_name, cat_name)] = votes[0].why

print(f"{SAMPLES} samples per cell, temperature 1.0\n")
print(f"{'task':<14}{'catalogue':<14}{'picks':<44}stable")
for key, counter in picks.items():
    spread = ", ".join(f"{name} x{n}" for name, n in counter.most_common())
    print(f"{key[0]:<14}{key[1]:<14}{spread:<44}{'yes' if len(counter) == 1 else 'NO'}")

print("\nthe reason given (first sample of each):")
for key, why in reasons.items():
    print(f"   {key[0]:<12}{key[1]:<12}{why[:96]}")

3 samples per cell, temperature 1.0

task          catalogue     picks                                       stable
unambiguous   as written    release-notes x3                            yes
unambiguous   degraded      release-notes x3                            yes
ambiguous     as written    incident-report x3                          yes
ambiguous     degraded      incident-report x3                          yes

the reason given (first sample of each):
   unambiguous as written  The user wants customer-facing release notes for a specific version based on merged changes.
   unambiguous degraded    The user wants customer-facing notes for a new version release.
   ambiguous   as written  The user is asking for customer-facing incident communication about a resolved export outage and
   ambiguous   degraded    The request is for customer-facing text about a resolved incident and its fix.


**Interpreting your results.** Compare the two catalogue versions for each task. For the unambiguous task, both may select the appropriate skill, even when the release-note description is vague. Check the explanations to see whether the selector refers to the description or relies on the skill’s name and the task wording.

For the ambiguous task, the request may not provide enough information to determine which output the user wants. The `stable` column shows whether the selector makes the same choice across repeated runs. Consistency alone, however, doesn’t establish that the choice is appropriate.

A vague `when_to_use` description can still produce a correct selection, but it makes the basis for that selection harder to assess. Explicit exclusions, such as “not for incident reports or upgrade guides”, help distinguish skills with overlapping purposes. These boundaries are easy to overlook when writing the first skill, because there are no competing skills yet.

Skill selection is also relatively inexpensive to test. A small set of task and expected-selection pairs can help detect routing changes as descriptions are revised or new skills are added. Include ambiguous requests in that set, with clarification as the expected outcome where the intended output is unclear. Asking the user should be an explicit routing option.

### An optional experiment: using a decision model for skill selection

In Section 3, we used a chat model to select a skill by generating its name. We repeated each request three times to examine selection stability, but the generated name itself doesn’t indicate how reliable the selection is. This optional experiment compares that approach with [Jev](https://openrouter.ai/typesafe/jev-router), TypeSafe’s System One decision model on OpenRouter, using a larger labelled task set.

Both selectors receive the same three options: the two skills and an explicit `clarify` option. This allows either selector to request clarification when the task doesn’t provide enough information to choose a skill. We label the ambiguous tasks as `clarify` because that’s the behavior we want from this harness. A different design could support producing both outputs.

Jev returns a probability for each option and a separate `confidence` value. We report the selected option’s probability and the confidence separately, since they shouldn’t be assumed to represent the same quantity.

The threshold used below is illustrative. We vary it across the same 12 tasks to examine how it affects selection and clarification. Choosing a threshold would require a separate development set, followed by evaluation on held-out tasks.

This experiment is optional, and later sections don’t depend on its results. Set `RUN_JEV_LIVE = False` to view it without making live requests.

In [ ]:
# --- an optional decision interface: TypeSafe's Jev on OpenRouter ----------------
# Jev is a System One model: given a state and typed questions it returns typed
# decisions with probabilities for software to act on, not prose. We point it at the
# chapter's own decision points. Its probability is a decision signal, not proof a
# rule was followed. One interface, reused in section 4 and section 8.
import json as _json, urllib.request as _url

JEV_MODEL = "typesafe/jev-1.13"                       # or "~typesafe/jev-latest"
DECISIONS_URL = "https://openrouter.ai/api/alpha/decisions"
RUN_JEV_LIVE = True                                   # False -> deterministic stand-ins, offline
ROUTE_TAU = 0.70                                      # illustrative, not tuned here
JEV_USAGE: List[Dict[str, object]] = []

# Both selectors choose from the SAME action set, so the comparison is a fair fight:
# the two skills, plus an explicit "clarify" the chat selector can pick too.
CLARIFY_DESC = ("The request is ambiguous or needs more than one skill; ask a "
                "clarifying question or route to both instead of committing to one.")
ROUTE_OPTIONS = {**AS_WRITTEN, "clarify": CLARIFY_DESC}

# A larger labelled set. On the ambiguous items the label is "clarify" — a design
# choice (ask, or produce both), NOT a claim that abstaining is the only right answer.
LABELLED = [
    ("Version 2.4.0 went out this morning. Here are the merged changes; write the notes our customers will see.", "release-notes"),
    ("We shipped 3.1.0 with a new scheduler view. Draft the customer release notes.", "release-notes"),
    ("Turn this changelog for 5.0 into customer-facing release notes.", "release-notes"),
    ("2.4.0 was released to all customers; write the release notes for what changed.", "release-notes"),
    ("The ingest API was down 14:02 to 15:07 UTC yesterday and is now resolved. Write the customer incident report.", "incident-report"),
    ("The postmortem for Tuesday's outage is done. Draft the customer-facing incident report.", "incident-report"),
    ("Write the incident report for last week's scheduler outage, now resolved.", "incident-report"),
    ("The exporter returned 500s for an hour this morning and is fixed. Customer incident writeup, please.", "incident-report"),
    ("Exports were failing for about an hour on Tuesday. We shipped 2.4.1 with the fix this morning — write the customer-facing text.", "clarify"),
    ("We had a partial outage and also released two features. Write something for customers.", "clarify"),
    ("There was an incident yesterday and version 2.5.0 shipped today. One customer message, please.", "clarify"),
    ("Customers want to know about the downtime and the new release. Draft the note.", "clarify"),
]

_REL = ("version", "shipped", "ship ", "release", "released", "changelog", "merged changes", "rolled out", "new feature")
_INC = ("outage", "down", "incident", "failing", "failed", "500s", "503", "postmortem", "resolved", "degraded", "unavailable", "downtime")


def _signals(task: str):
    t = task.lower()
    return sum(k in t for k in _REL), sum(k in t for k in _INC)


def _jev_live(body):
    t0 = time.perf_counter()
    req = _url.Request(DECISIONS_URL, data=_json.dumps(body).encode(),
                       headers={"Authorization": f"Bearer {os.environ['OPENROUTER_API_KEY']}",
                                "Content-Type": "application/json"})
    with _url.urlopen(req, timeout=60) as r:
        resp = _json.load(r)
    resp.setdefault("usage", {})["seconds"] = time.perf_counter() - t0
    return resp


def _jev_replay(body):
    """Deterministic stand-in with the live response shape, for offline rendering only.
    Numbers are illustrative; a live run produces the real decisions and probabilities."""
    state, answers = body["state"], {}
    for name, q in body["questions"].items():
        crit = q.get("criteria", {})
        if q["type"] == "choice":
            rel, inc = _signals(str(state.get("task", "")))
            if rel and inc:                                   # both -> near tie, some clarify mass
                base = {"release-notes": 0.42 if rel >= inc else 0.38,
                        "incident-report": 0.40 if rel >= inc else 0.44, "clarify": 0.18}
            elif rel:
                base = {"release-notes": 0.90, "incident-report": 0.05, "clarify": 0.05}
            elif inc:
                base = {"release-notes": 0.05, "incident-report": 0.90, "clarify": 0.05}
            else:
                base = {"release-notes": 0.15, "incident-report": 0.15, "clarify": 0.70}
            probs = {o: round(base.get(o, 0.0), 3) for o in crit}
            s = sum(probs.values()) or 1.0
            probs = {o: round(p / s, 3) for o, p in probs.items()}
            top = sorted(probs.values(), reverse=True)
            answers[name] = {"type": "choice", "choice": max(probs, key=probs.get),
                             "confidence": round(top[0] - (top[1] if len(top) > 1 else 0), 2),
                             "probabilities": probs}
        elif q["type"] == "noul":
            instr = q["instructions"].lower()
            sent = str(state.get("sentence", "")).lower()
            facts = " ".join(state.get("facts", [])).lower()
            perf = any(w in sent for w in ("fast", "faster", "speed", "throughput", "memory", "latency"))
            mags = re.findall(r"\d+\s*(?:percent|%)|\btwice\b|\btwo times\b|\b\d+x\b|\bhalf\b|\b\d+\b", sent)
            if "overlap" in instr:                            # boundary review
                rel, inc = _signals(str(state.get("task", "")))
                # ground truth: the ambiguous task DOES overlap and should flag; the
                # saved live run scored it low (a false negative), reproduced here.
                answers[name] = {"type": "noul", "noul": 0.34 if (rel and inc) else 0.12}
            elif "state" in instr:                            # does the OUTPUT state the figure?
                answers[name] = {"type": "noul", "noul": 0.97 if (not perf or mags) else 0.05}
            else:                                             # supported-by-facts claim check
                if not perf:
                    p_sup = 0.97                              # not a performance claim: nothing to support
                elif not mags:
                    # vague claim ("much faster"): plausibly supported if facts measure that
                    # dimension at all -- which is why "supported" alone is not enough and the
                    # states-figure question is asked too.
                    p_sup = 0.68 if re.search(r"\d|twice|percent|half", facts) else 0.20
                else:
                    p_sup = 0.95 if all(m.strip() in facts for m in mags) else 0.08
                answers[name] = {"type": "noul", "noul": round(p_sup, 2)}
    return {"id": "replay", "model": JEV_MODEL, "answers": answers,
            "usage": {"input_tokens": 0, "output_tokens": 0, "cost": 0.0, "seconds": 0.0}}


def jev_decide(state, questions, label):
    body = {"model": JEV_MODEL, "state": state, "questions": questions}
    resp = _jev_live(body) if RUN_JEV_LIVE else _jev_replay(body)
    u = resp.get("usage", {})
    JEV_USAGE.append({"label": label, "input_tokens": u.get("input_tokens", 0),
                      "cost": u.get("cost", 0.0), "seconds": u.get("seconds", 0.0)})
    return resp["answers"]


if RUN_JEV_LIVE and not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key: ")


# The two selectors, given the SAME options and the SAME task.
def jev_route(task, label):
    return jev_decide({"task": task},
                      {"route": {"type": "choice", "instructions": "Which skill should handle this task?",
                                 "criteria": dict(ROUTE_OPTIONS)}}, label)["route"]


def chat_route(task, label):
    return typed_call(model=ROUTER, response_model=SkillChoice, temperature=0.0, label=label,
                      messages=[{"role": "system", "content":
                                 "Choose exactly one option for the task. Answer with its name."},
                                {"role": "user", "content":
                                 f"OPTIONS\n{catalogue(ROUTE_OPTIONS)}\n\nTASK\n{task}"}]).skill


# Jev's action under an illustrative threshold: it can abstain by picking clarify OR
# by low confidence in a skill. Report the selected option's probability separately
# from the confidence field — they need not be the same quantity.
def jev_action(ans):
    if ans["choice"] == "clarify" or ans["confidence"] < ROUTE_TAU:
        return "clarify"
    return ans["choice"]


print(f"ROUTE_TAU = {ROUTE_TAU} (illustrative; tune on held-out tasks)\n")
print(f"  {'label':<16}{'chat pick':<17}{'jev choice':<17}{'p(choice)':<11}{'confidence':<12}{'jev action'}")
DECISIONS = []
for task, label in LABELLED:
    j = jev_route(task, "jev-route")
    c = chat_route(task, "chat-route")
    DECISIONS.append((task, label, c, j))
    pc = j["probabilities"].get(j["choice"], float("nan"))
    print(f"  {label:<16}{c:<17}{j['choice']:<17}{pc:<11.2f}{j['confidence']:<12.2f}{jev_action(j)}")

ROUTE_TAU = 0.7 (illustrative; tune on held-out tasks)

  label           chat pick        jev choice       p(choice)  confidence  jev action
  release-notes   release-notes    release-notes    1.00       1.00        release-notes
  release-notes   release-notes    release-notes    1.00       1.00        release-notes
  release-notes   release-notes    release-notes    1.00       1.00        release-notes
  release-notes   release-notes    release-notes    0.99       0.98        release-notes
  incident-report incident-report  incident-report  1.00       1.00        incident-report
  incident-report incident-report  incident-report  1.00       1.00        incident-report
  incident-report incident-report  incident-report  1.00       1.00        incident-report
  incident-report incident-report  incident-report  1.00       1.00        incident-report
  clarify         incident-report  incident-report  0.98       0.98        incident-report
  clarify         clarify          clarify     

In [ ]:
# Interpretation is COMPUTED from the run above, not asserted — on a live run the
# numbers change, so the sentences must follow the data.
skills = set(AS_WRITTEN)
decisive = [(t, lab, c, j) for (t, lab, c, j) in DECISIONS if lab in skills]
ambiguous = [(t, lab, c, j) for (t, lab, c, j) in DECISIONS if lab == "clarify"]

chat_dec = sum(c == lab for (_, lab, c, j) in decisive)
jev_dec = sum(j["choice"] == lab for (_, lab, c, j) in decisive)            # argmax, same action set
jev_dec_act = sum(jev_action(j) == lab for (_, lab, c, j) in decisive)      # after threshold

chat_amb_clarify = sum(c == "clarify" for (_, lab, c, j) in ambiguous)
jev_amb_clarify = sum(jev_action(j) == "clarify" for (_, lab, c, j) in ambiguous)
agree = sum(c == j["choice"] for (_, lab, c, j) in DECISIONS)


def _usage(prefix):
    rows = [u for u in USAGE if u["label"].startswith(prefix)]
    return len(rows), sum(u["seconds"] for u in rows), \
           sum(u.get("prompt_tokens", 0) + u.get("completion_tokens", 0) for u in rows)


cc, cs, ct = _usage("chat-route")
jc = len(JEV_USAGE); js = sum(u["seconds"] for u in JEV_USAGE); jcost = sum(u["cost"] for u in JEV_USAGE)

nd, na = len(decisive), len(ambiguous)
print(f"  decisive tasks (n={nd}):")
print(f"    chat selector, argmax == label ......... {chat_dec}/{nd}")
print(f"    jev, argmax == label ................... {jev_dec}/{nd}")
print(f"    jev, action-after-threshold == label ... {jev_dec_act}/{nd}")
print(f"  ambiguous tasks (n={na}, label 'clarify' — a policy choice, not sole ground truth):")
print(f"    chat selector chose clarify ............ {chat_amb_clarify}/{na}")
print(f"    jev clarified (clarify pick or conf<tau) {jev_amb_clarify}/{na}")
print(f"  the two selectors agreed on the argmax ... {agree}/{len(DECISIONS)}")
print(f"  cost signal: chat {cc} calls, {ct} tok (in+out), {cs:.1f}s   |   "
      f"jev {jc} calls, {jcost:.6f} USD (in only), {js:.1f}s")



  decisive tasks (n=8):
    chat selector, argmax == label ......... 8/8
    jev, argmax == label ................... 8/8
    jev, action-after-threshold == label ... 8/8
  ambiguous tasks (n=4, label 'clarify' — a policy choice, not sole ground truth):
    chat selector chose clarify ............ 3/4
    jev clarified (clarify pick or conf<tau) 3/4
  the two selectors agreed on the argmax ... 11/12
  cost signal: chat 12 calls, 3240 tok (in+out), 10.8s   |   jev 12 calls, 0.000225 USD (in only), 2.1s


**How to read this comparison.** Each selector gets the same task and the same three options in one call. The table reports decisions and measured latency for this run. Jev supplies an option distribution and a separate confidence value, which the harness can use to choose `clarify`; these 12 items do not establish calibration or a general accuracy advantage. Offline replay values are illustrative.


`ROUTE_TAU` is a policy knob the sweep below is an
illustration on the same 12 items to show how the knob behaves. A real threshold is chosen on a held-out split and reported there.

In [ ]:
skills = set(AS_WRITTEN)
print(f"  {'tau':<6}{'decisive routed right':<24}{'ambiguous clarified':<22}")
for tau in (0.20, 0.35, 0.50, 0.70, 0.85):
    dec_ok = amb_clar = nd = na = 0
    for (t, lab, c, j) in DECISIONS:
        act = "clarify" if (j["choice"] == "clarify" or j["confidence"] < tau) else j["choice"]
        if lab in skills:
            nd += 1; dec_ok += act == lab
        else:
            na += 1; amb_clar += act == "clarify"
    print(f"  {tau:<6}{f'{dec_ok}/{nd}':<24}{f'{amb_clar}/{na}':<22}")


  tau   decisive routed right   ambiguous clarified   
  0.2   8/8                     2/4                   
  0.35  8/8                     2/4                   
  0.5   8/8                     3/4                   
  0.7   8/8                     3/4                   
  0.85  8/8                     3/4                   


The sweep reuses the 12 tasks above. It illustrates how the routing policy changes as `tau` changes; it is not a tuned or held-out result. To estimate performance, choose the threshold on development tasks and then report it on untouched evaluation tasks.


In [ ]:
# Boundary review (section 4), as a bounded question. Section 4's overlap() compares
# ownership *claims* statically — two skills naming one section id. The complementary
# question is semantic and per task: for this task, do two when_to_use descriptions
# actually compete? Compare the answer with labelled tasks, including misses.
def jev_boundary_overlap(task, a, b, label):
    return jev_decide(
        {"task": task, "skill_a": AS_WRITTEN[a], "skill_b": AS_WRITTEN[b]},
        {"overlap": {"type": "noul",
                     "instructions": "Do skill_a and skill_b both plausibly apply to this task (their when_to_use overlap here)?",
                     "criteria": {"true": "Both descriptions fit the task; the boundary is ambiguous.",
                                  "false": "Only one description fits; the boundary is clear."}}},
        label)["overlap"]["noul"]


REVIEW_TAU = 0.50   # also illustrative
sample = [LABELLED[0], LABELLED[4], LABELLED[8]]   # one release, one incident, one ambiguous
boundary_results = []
for task, label in sample:
    p = jev_boundary_overlap(task, "release-notes", "incident-report", f"jev-overlap[{label}]")
    boundary_results.append((label, p))
    print(f"  [{label:<15}] p(overlap)={p:.2f}  ->  {'FLAG for boundary review' if p >= REVIEW_TAU else 'boundary clear'}")


  [release-notes  ] p(overlap)=0.03  ->  boundary clear
  [incident-report] p(overlap)=0.03  ->  boundary clear
  [clarify        ] p(overlap)=0.11  ->  boundary clear


**Evaluating the boundary check.** The static `overlap()` check below identifies sections assigned to more than one skill. Jev instead assesses whether the skill boundaries are clear for a given task.

In the saved live run, Jev returned “boundary clear” with a low overlap probability for a task labelled `clarify`. Under our routing policy, this is a false negative: the checker failed to identify a case requiring clarification. Results may vary across runs, but this failure should remain part of the evaluation. Test the checker against labelled cases before using it to control routing.

## 4. Boundaries: two skills, one section id

Both skills own a section called `summary.style`, and both own
`output.contract`, `prohibitions.hard` and `typography`. Separately, each is
coherent. The collision only exists if something loads both for one task — which
is exactly what a well-meaning "the agent should have all its skills available"
change does.

The check below is nine lines and runs at load time. What it produces is not a warning to be triaged later: a task that needs two overlapping skills has a design error in the skills, and the fix is to name the sections so that ownership is unambiguous (`release_notes.summary` vs `incident.summary`), or to split the shared rule out into an artifact both can load.


In [ ]:
def overlap(artifacts: Sequence[Artifact]) -> Dict[str, List[str]]:
    """Section ids claimed by more than one artifact."""
    claims: Dict[str, List[str]] = {}
    for a in artifacts:
        for section in a.owns:
            claims.setdefault(section, []).append(a.name)
    return {sid: names for sid, names in claims.items() if len(names) > 1}


def load_for_task(names: Sequence[str]) -> List[Artifact]:
    """Identity + project + the named artifacts, refusing an ambiguous surface."""
    chosen = ALWAYS + [ARTIFACTS[n] for n in names]
    clash = overlap(chosen)
    if clash:
        raise ValueError(
            "ambiguous surface — every section needs exactly one owner:\n" +
            "\n".join(f"   {sid}: claimed by {', '.join(owners)}"
                      for sid, owners in sorted(clash.items())))
    return chosen


print("one skill for the task:")
surface = load_for_task(["release-notes", "plainspoken"])
print(f"   loaded: {', '.join(a.name for a in surface)}\n")

print("both skills, 'just in case':")
try:
    load_for_task(["release-notes", "incident-report", "plainspoken"])
except ValueError as exc:
    print(textwrap.indent(str(exc), "   "))


one skill for the task:
   loaded: acme-docs-agent, acme-data-platform, release-notes, plainspoken

both skills, 'just in case':
   ambiguous surface — every section needs exactly one owner:
      output.contract: claimed by release-notes, incident-report
      prohibitions.hard: claimed by release-notes, incident-report
      summary.style: claimed by release-notes, incident-report
      typography: claimed by release-notes, incident-report


The check compares ownership claims, not the instruction text. Two artifacts may contain the same instruction, but each named rule needs a single authoritative source.

## 5. Skills, tools, personas, workflows


| | What it is | Changes the output contract? | Swappable mid-task? |
|---|---|---|---|
| **Tool** | a function the agent can call, with a typed signature | no — it supplies facts | yes |
| **Persona** | how the text sounds | no | yes |
| **Skill** | what the artifact *is*: structure, counts, prohibitions | **yes** | no — changing it changes the task |
| **Workflow** | the order in which skills and tools run | no, per step | it *is* the order |

The cell below demonstrates the two rows that matter. The tool is the thing that
fetches the merged changes, and the harness calls it, and its result is *input*, not instruction. Then the persona is swapped under a fixed skill: the sentences change, the contract does not.


In [ ]:
# A tool: typed, boring, and the only thing here that talks to the outside world.
def merged_changes(version: str) -> Dict[str, object]:
    """Pretend this reads the merge queue. Facts in, no instructions."""
    return {
        "version": version,
        "breaking": True,                       # the harness knows; never inferred
        "changes": [
            "the CSV exporter was rewritten; exports run about twice as fast",
            "fixed escaping of quotes inside CSV fields",
            "peak memory during export reduced by roughly 40 percent",
            "the /v1/export endpoint now returns 202 with a job id instead of the "
            "file; callers must poll /v1/export/{id}",
        ],
    }


FACTS = merged_changes("2.4.0")
for c in FACTS["changes"]:
    print(f"   - {c}")
print(f"   breaking={FACTS['breaking']}")


   - the CSV exporter was rewritten; exports run about twice as fast
   - fixed escaping of quotes inside CSV fields
   - peak memory during export reduced by roughly 40 percent
   - the /v1/export endpoint now returns 202 with a job id instead of the file; callers must poll /v1/export/{id}
   breaking=True


The tool returns facts for the harness to supply to the model. The `breaking` flag is owned by the harness and is not inferred from the release-note skill.


## 6. Runtime scaffolding: compiling instructions for a specific task

The next step is to assemble the instructions for the current task. Here, the task specifies `tone=casual, audience=customer`, while the skill file includes two tone variants, two audience variants, and four worked examples. Only the applicable instructions and examples need to be included. Loading the other variants adds unnecessary context and may introduce conflicting guidance, such as a formal example alongside an instruction to use a casual tone.

The compiler performs four steps:

1. **Select** the required artifacts. Identity and project instructions are always included, along with the selected persona and skill.
2. **Filter** the variants within each section. For this task, the compiler keeps `### Tone: casual` and removes `### Tone: formal`. A heading can specify multiple conditions, such as `### Tone: casual, Audience: customer`, in which case all conditions must match.
3. **Deduplicate** repeated rule text. When the same rule appears in multiple artifacts, the compiler keeps the first occurrence according to load order: identity, project, skill, then persona. This gives durable instructions precedence over identical task-specific restatements. Section ownership doesn’t determine which copy is retained: both artifacts may legitimately own different sections containing the same text. Section 4 checks overlapping section IDs; this step removes repeated text. Each removal is recorded with the artifact containing the retained copy.
4. **Emit** the assembled instruction string and a manifest recording what was included.

This compilation happens before the model call. The selection, filtering, and deduplication logic is explicit in the code, and the manifest can be attached to the execution trace so we can inspect which instructions the model received.

In [ ]:
_H3 = re.compile(r"^### (.+)$", re.M)
AXIS_LABELS = {"tone": "Tone", "audience": "Audience"}


def _heading_axes(heading: str) -> Dict[str, str]:
    """`Tone: casual, Audience: customer` -> {'tone': 'casual', 'audience': 'customer'}."""
    found = {}
    for part in heading.split(","):
        if ":" not in part:
            continue
        key, _, value = part.partition(":")
        key = key.strip().lower()
        if key in AXIS_LABELS:
            found[key] = value.strip().lower()
    return found


def slice_section(body: str, axes: Dict[str, str]) -> str:
    """Keep the preamble, and only the H3 blocks whose axes all match."""
    marks = list(_H3.finditer(body))
    if not marks:
        return body
    kept = [body[:marks[0].start()].strip()]
    for i, mark in enumerate(marks):
        end = marks[i + 1].start() if i + 1 < len(marks) else len(body)
        heading, block = mark.group(1).strip(), body[mark.end():end].strip()
        wanted = _heading_axes(heading)
        # A heading with no axes at all is unconditional prose, so it stays.
        if wanted and any(axes.get(k) != v for k, v in wanted.items()):
            continue
        kept.append(block)
    return "\n\n".join(x for x in kept if x)


def _norm(line: str) -> str:
    return re.sub(r"\s+", " ", line.strip().lower())


#: Load order, and therefore precedence when the same paragraph appears twice.
#: Durable first: a rule the agent carries everywhere outranks a task-local
#: restatement of it, so the surviving copy is the one with the wider scope.
KIND_PRECEDENCE = ("identity", "project", "skill", "persona")


def in_load_order(artifacts: Sequence[Artifact]) -> List[Artifact]:
    return sorted(artifacts, key=lambda a: KIND_PRECEDENCE.index(a.kind))


@dataclass
class Surface:
    text: str
    manifest: Dict[str, object]

    @property
    def words(self) -> int:
        return len(self.text.split())


def compile_surface(*, artifacts: Sequence[Artifact], axes: Dict[str, str],
                    facts: Dict[str, object]) -> Surface:
    blocks, seen, dropped, used = [], {}, [], []
    for a in in_load_order(artifacts):
        for sid, body in a.sections.items():
            text = slice_section(body, axes) if a.axes else body
            # Deduplicate paragraph by paragraph, not section by section: the
            # same sentence written by two teams is the common case, two
            # identical sections is not. The copy that survives is the one from
            # the artifact earliest in load order, and the manifest records who
            # kept it — so "who says this?" has exactly one answer.
            kept = []
            for para in re.split(r"\n\s*\n", text):
                if not para.strip():
                    continue
                key = _norm(para)
                if key in seen:
                    dropped.append({"dropped_from": f"{a.name}:{sid}",
                                    "kept_from": seen[key],
                                    "text": para.strip()[:70]})
                    continue
                seen[key] = f"{a.name}:{sid}"
                kept.append(para.strip())
            if not kept:
                continue
            blocks.append("\n\n".join(kept))
            used.append({"artifact": a.name, "version": a.version,
                         "digest": a.digest, "section": sid})
    fact_lines = "\n".join(f"- {k}: {v}" for k, v in facts.items() if k != "changes")
    blocks.append(f"## Facts supplied by the harness\n\n{fact_lines}\n\n"
                  "These are given. Do not restate them as claims of your own, and "
                  "do not infer any fact that is not listed.")
    text = "\n\n".join(blocks)
    return Surface(text=text, manifest={
        "axes": dict(axes), "sections": used, "deduplicated": dropped,
        "surface_digest": hashlib.sha256(text.encode()).hexdigest()[:12]})


AXES = {"tone": "casual", "audience": "customer"}
SURFACE = compile_surface(artifacts=surface, axes=AXES,
                          facts={k: v for k, v in FACTS.items() if k != "changes"})

print(f"axes: {AXES}   surface digest: {SURFACE.manifest['surface_digest']}")
print(f"{len(SURFACE.manifest['sections'])} sections, {SURFACE.words} words")
for d in SURFACE.manifest["deduplicated"]:
    print(f"   deduplicated  {d['text']}")
    print(f"{'':17}kept from {d['kept_from']}, dropped from {d['dropped_from']}")
print()
print(SURFACE.text[:1600])
print("\n[...]")


axes: {'tone': 'casual', 'audience': 'customer'}   surface digest: 61368dc1fa99
14 sections, 584 words
   deduplicated  Version numbers are written in digits, never in words: `2.4.0`.
                 kept from acme-data-platform:project.versioning, dropped from release-notes:typography

## Who you are

You are the documentation agent for the Acme data platform. You write text that
ships to customers under Acme's name: release notes, incident reports, upgrade
guides. You do not write marketing copy, and you do not answer support tickets.

## What you never do

- Never state a number, date, version or endpoint that is not in the input.
- Never promise a future release, a roadmap item or a deprecation date.
- Never describe the impact on a named customer.

These hold in every task, under every persona, in every skill. A skill may add
prohibitions; it may not remove one of these.

## When you stop

If the input contradicts itself, or a required fact is missing, return the
partial result a

Read the compiled text and notice what is *not* there: no formal bullet rule, no developer-only prohibition about unmeasured performance claims, and one worked example instead of four. The skill file still contains all of them, this is a per-request omission, not an edit. That distinction matters the first time an SME asks why their rule "disappeared".


## 7. Evaluating runtime scaffolding

To assess the effect of compilation, we compare two calls using the same model, input, and task. One receives all the maintained instructions, while the other receives only the compiled instructions for that task. Both are asked to return the same typed object so that we can compare their outputs.

The table reports two measurements: **prompt tokens** and **latency**. Prompt tokens decrease because the compiler removes variants that don’t apply. Identity instructions, project facts, and unconditional rules remain in both calls, so the reduction depends on how much of the original content is conditional. This example contains relatively little variant content. As a skill adds more tones, audiences, and other conditions, more content may be excluded from any individual call.

Latency decreases less than the prompt token count because processing the prompt accounts for only part of the response time. Measuring both helps us distinguish a reduction in context size from its effect on execution time. Chapter 4 examines a different mechanism for improving execution efficiency.


In [ ]:
try:
    import tiktoken
    _enc = tiktoken.get_encoding("cl100k_base")
    def ntok(t): return len(_enc.encode(t))
    counter = "tiktoken cl100k_base"
except Exception:
    def ntok(t): return max(1, round(len(t) / 4))
    counter = "approximation (chars/4)"


def everything_surface(artifacts: Sequence[Artifact], facts: Dict[str, object]) -> Surface:
    """Every artifact, every variant — the way a hand-maintained prompt grows."""
    blocks, used = [], []
    for a in artifacts:
        for sid, body in a.sections.items():
            blocks.append(body)
            used.append({"artifact": a.name, "version": a.version,
                         "digest": a.digest, "section": sid})
    fact_lines = "\n".join(f"- {k}: {v}" for k, v in facts.items() if k != "changes")
    blocks.append(f"## Facts supplied by the harness\n\n{fact_lines}")
    text = "\n\n".join(blocks)
    return Surface(text=text, manifest={"axes": {}, "sections": used, "deduplicated": [],
                                        "surface_digest": hashlib.sha256(text.encode()).hexdigest()[:12]})


EVERYTHING = everything_surface(surface, {k: v for k, v in FACTS.items() if k != "changes"})

print(f"instruction surface ({counter})")
print(f"   {'everything, all variants':<28}{ntok(EVERYTHING.text):>7} tokens  "
      f"{EVERYTHING.words:>5} words  {len(EVERYTHING.manifest['sections']):>2} sections")
print(f"   {'compiled for this task':<28}{ntok(SURFACE.text):>7} tokens  "
      f"{SURFACE.words:>5} words  {len(SURFACE.manifest['sections']):>2} sections")
print(f"   {'reduction':<28}{1 - ntok(SURFACE.text) / ntok(EVERYTHING.text):>6.0%}")


instruction surface (tiktoken cl100k_base)
   everything, all variants       1177 tokens    837 words  14 sections
   compiled for this task          788 tokens    584 words  14 sections
   reduction                      33%


In [ ]:
class ReleaseNotes(BaseModel):
    summary: str = Field(description="One sentence, at most 25 words.")
    highlights: List[str] = Field(description="Three to six bullets, one change each.")


USER_INPUT = (f"Release {FACTS['version']} of the Acme data platform.\n"
              + "\n".join(f"- {c}" for c in FACTS["changes"]))


def write_notes(surface_: Surface, label: str) -> ReleaseNotes:
    return typed_call(model=WRITER, response_model=ReleaseNotes, temperature=0.3,
                      label=label,
                      messages=[{"role": "system", "content": surface_.text},
                                {"role": "user", "content": USER_INPUT}])


RESULTS = {}
for label, s in (("everything", EVERYTHING), ("compiled", SURFACE)):
    RESULTS[label] = write_notes(s, f"write[{label}]")

rows = {u["label"]: u for u in USAGE if u["label"].startswith("write[")}
print(f"{'run':<22}{'prompt tok':>11}{'out tok':>9}{'seconds':>9}")
for label in ("everything", "compiled"):
    u = rows[f"write[{label}]"]
    print(f"{label:<22}{u['prompt_tokens']:>11}{u['completion_tokens']:>9}{u['seconds']:>9.1f}")

for label, notes in RESULTS.items():
    print(f"\n--- {label}")
    print(f"  summary  : {notes.summary}")
    for h in notes.highlights:
        print(f"  highlight: {h}")


run                    prompt tok  out tok  seconds
everything                   1344      349      4.7
compiled                      953      305      5.0

--- everything
  summary  : Version 2.4.0 rewrites the CSV exporter and changes `/v1/export` to return a job id instead of the file.
  highlight: Improved export throughput, approximately two times
  highlight: Reduced peak memory during export by roughly 40 percent
  highlight: Fixed escaping of quotes inside CSV fields
  highlight: Revised `/v1/export` response: `202` with a job id, followed by polling at `/v1/export/{id}`

--- compiled
  summary  : We rebuilt the CSV exporter, so exports run about twice as fast.
  highlight: Export about twice as fast
  highlight: Handle quotes correctly inside CSV fields
  highlight: Use roughly 40 percent less peak memory during export
  highlight: Poll export jobs after starting an export


## 8. The skill as a testable unit

A prompt is reviewed. A skill is *tested* — and the tests split cleanly in two,
which is the practical reason the compile step exists at all.

**Tests on the surface** need no model. They run in CI, in milliseconds, and they
are the ones that catch the failure nobody notices in review: a casual rule
present in a formal surface, a contract that no longer matches the schema the
harness parses into, a section that lost its owner.

**Tests on the output** need a model and are therefore slower, noisier and
non-deterministic — so they should only carry the assertions that genuinely need
generated text. Counts and forbidden words are not in that category.

The second cell then does what happens in real life: an SME edits the skill, the
edit is entirely reasonable, and a surface test fails.


In [ ]:
def surface_tests(surface_: Surface, axes: Dict[str, str]) -> List[Tuple[str, bool, str]]:
    """Assertions about the compiled instruction surface. No model involved."""
    text, lower = surface_.text, surface_.text.lower()
    results = []

    # 1. The wrong variant must not be present at all. Each tone gets a marker
    #    phrase that appears in its rules and nowhere else, so the check is the
    #    same check in both directions.
    MARKERS = {"formal": ("noun phrase", "first-person plural subject"),
               "casual": ("starts with a verb", "we rebuilt")}
    other_tone = "formal" if axes["tone"] == "casual" else "casual"
    leaked = [m for m in MARKERS[other_tone] if m in lower]
    results.append((f"no {other_tone}-tone rule in the surface", not leaked,
                    f"found {leaked}"))

    # 2. The contract in the skill and the schema we parse into must agree.
    declared = set(re.findall(r"^-\s+`(\w+)`:", surface_.text, re.M))
    expected = set(ReleaseNotes.model_fields)
    results.append(("contract keys match the response model", declared == expected,
                    f"skill={sorted(declared)} model={sorted(expected)}"))

    # 3. The refusals survive assembly, whatever else was sliced away.
    results.append(("identity refusals present",
                    "never state a number" in text.lower(), ""))

    # 4. Every section in the surface is attributable to an artifact and version.
    manifest_ok = all(s.get("digest") and s.get("version")
                      for s in surface_.manifest["sections"])
    results.append(("every section carries provenance", manifest_ok, ""))
    return results


def report(results):
    for name, ok, detail in results:
        print(f"   [{'ok' if ok else 'FAIL'}] {name}{'' if ok else ' — ' + detail}")
    return all(ok for _, ok, _ in results)


print("surface tests (no model):")
report(surface_tests(SURFACE, AXES))

print("\noutput tests (the model's text):")
notes = RESULTS["compiled"]
checks = [
    ("three to six highlights", 3 <= len(notes.highlights) <= 6, str(len(notes.highlights))),
    ("summary at most 25 words", len(notes.summary.split()) <= 25,
     str(len(notes.summary.split()))),
    ("no exclamation marks", "!" not in notes.summary + " ".join(notes.highlights), ""),
    ("no superlative", not re.search(r"\b(best|fastest|most powerful)\b",
                                     notes.summary + " ".join(notes.highlights), re.I), ""),
]
report(checks)


surface tests (no model):
   [ok] no formal-tone rule in the surface
   [ok] contract keys match the response model
   [ok] identity refusals present
   [ok] every section carries provenance

output tests (the model's text):
   [ok] three to six highlights
   [ok] summary at most 25 words
   [ok] no exclamation marks
   [ok] no superlative


True

In [ ]:
# An SME adds a helpful example to the top of the highlights section. It reads
# fine in review — but it sits in the section preamble, which every variant
# keeps, and the example itself is written in the casual voice. It now ships
# inside the FORMAL surface as well.
ORIGINAL = ARTIFACTS["release-notes"].sections["highlights.structure"]
edited = ("For example: `we rebuilt the exporter, so exports are twice as fast`.\n\n"
          + ORIGINAL)
assert "we rebuilt" in edited, "the demo edit did not apply"

patched = Artifact(**{**ARTIFACTS["release-notes"].__dict__,
                      "sections": {**ARTIFACTS["release-notes"].sections,
                                   "highlights.structure": edited}})
broken = compile_surface(artifacts=[a if a.name != "release-notes" else patched
                                    for a in surface],
                         axes={"tone": "formal", "audience": "customer"},
                         facts={"version": FACTS["version"], "breaking": FACTS["breaking"]})

print("the edit, as it looks in review:")
print(f"   + {edited.splitlines()[0]}\n")
print("after the edit, compiling the FORMAL surface:")
ok = report(surface_tests(broken, {"tone": "formal", "audience": "customer"}))
print(f"\nCI result: {'pass' if ok else 'fail — the edit does not ship'}")


the edit, as it looks in review:
   + For example: `we rebuilt the exporter, so exports are twice as fast`.

after the edit, compiling the FORMAL surface:
   [FAIL] no casual-tone rule in the surface — found ['we rebuilt']
   [ok] contract keys match the response model
   [ok] identity refusals present
   [ok] every section carries provenance

CI result: fail — the edit does not ship


**Checking a semantic requirement with the same decision interface.** Counts, forbidden words, and provenance can be checked deterministically in code and included in CI. The developer rule “no performance claim without the number that was measured” also requires us to compare the generated text with the source facts.

We evaluate two questions separately: is the performance claim supported by the release facts, and does the sentence include the measured figure? A supported claim can still violate the rule if it omits the number, so both conditions must pass. The returned probabilities help determine which content to send back for correction, but they don’t guarantee that the rule was followed.

In [ ]:
# Output checking with the same interface (section 8). The surface tests above stay
# code -- counts, forbidden words, provenance are deterministic and belong in CI. One
# rule needs judgment, and it has TWO parts the developer prohibition folds together:
# "no performance claim without the number that was measured." That is really two
# questions, so we ask them separately:
#   1. Is the claim supported by the release facts?      (is it true?)
#   2. Does the sentence itself state the measured figure? (is it stated?)
# The output must pass BOTH. A sentence can be true yet still break the rule by not
# stating the number, and a sentence can state a number the facts do not support.
CLAIM_FACTS = FACTS["changes"]   # the merged changes the harness owns (section 5 tool result)


def jev_claim_checks(sentence, facts, label):
    ans = jev_decide(
        {"sentence": sentence, "facts": facts},
        {"supported": {"type": "noul",
                       "instructions": "Is every performance claim in the sentence supported by a measured figure present in facts?",
                       "criteria": {"true": "Any speed/memory/throughput claim matches a figure in facts, or there is no performance claim.",
                                    "false": "The sentence claims a performance improvement that facts do not measure."}},
         "states_figure": {"type": "noul",
                       "instructions": "If the sentence makes a performance claim, does the sentence itself state the measured number?",
                       "criteria": {"true": "The sentence gives the figure (a percentage or magnitude), or makes no performance claim.",
                                    "false": "The sentence claims a performance improvement in words with no number."}}},
        label)
    return ans["supported"]["noul"], ans["states_figure"]["noul"]


CLAIM_TAU = 0.50
candidates = [
    "Exports are now much faster.",                 # true in facts, but states no figure -> fails Q2
    "Exports run about twice as fast.",             # supported by facts AND states the figure -> passes
    "Exports use 60 percent less memory.",          # states a figure, but facts say 40 -> fails Q1
    "Fixed escaping of quotes inside CSV fields.",  # no performance claim -> passes both
]
for f in CLAIM_FACTS:
    print(f"   - {f}")
print(f"  {'p(supported)':<14}{'p(states figure)':<18}{'action':<24}sentence")
for s in candidates:
    p_sup, p_fig = jev_claim_checks(s, CLAIM_FACTS, "jev-claim")
    ok = p_sup >= CLAIM_TAU and p_fig >= CLAIM_TAU
    action = "ok" if ok else ("SEND FOR CORRECTION" if p_sup < CLAIM_TAU else "SEND: state the figure")
    print(f"  {p_sup:<14.2f}{p_fig:<18.2f}{action:<24}{s}")


   - the CSV exporter was rewritten; exports run about twice as fast
   - fixed escaping of quotes inside CSV fields
   - peak memory during export reduced by roughly 40 percent
   - the /v1/export endpoint now returns 202 with a job id instead of the file; callers must poll /v1/export/{id}
  p(supported)  p(states figure)  action                  sentence
  0.64          0.06              SEND: state the figure  Exports are now much faster.
  0.81          0.79              ok                      Exports run about twice as fast.
  0.07          0.97              SEND FOR CORRECTION     Exports use 60 percent less memory.
  0.94          0.90              ok                      Fixed escaping of quotes inside CSV fields.


The two questions isolate different failures. In the saved live run, “much faster” had support in the facts but omitted the measured magnitude; “60 percent” stated a magnitude that conflicted with the facts. Both are sent for correction.


## 9. Versioning artifacts and compiled instructions

Each artifact has a version and a content digest. The compiled instructions also have a digest, calculated from the assembled text. These values serve different purposes:

- The **artifact version** records an explicitly declared revision that people can review and discuss.
- The **artifact digest** identifies the artifact’s content, allowing us to detect changes even when someone forgets to update the version.
- The **compiled instruction digest** identifies the exact instruction text supplied to the model. The same artifact versions can produce different compiled instructions when task variables, such as tone or audience, change.

Recording the compiled instruction digest and manifest in the trace lets us investigate which instructions a model received when reviewing its output later. Chapter 4 uses these records to build an export that links each execution to the artifacts and compilation choices used for that run.

In [ ]:
VARIANTS = [{"tone": t, "audience": a} for t in ("formal", "casual")
            for a in ("customer", "developer")]

print(f"{'tone':10}{'audience':12}{'words':>7}{'tokens':>8}  surface digest")
for axes in VARIANTS:
    s = compile_surface(artifacts=surface, axes=axes,
                        facts={"version": FACTS["version"], "breaking": FACTS["breaking"]})
    print(f"{axes['tone']:10}{axes['audience']:12}{s.words:>7}{ntok(s.text):>8}  "
          f"{s.manifest['surface_digest']}")

print(f"\n{'everything, all variants':<22}{EVERYTHING.words:>7}{ntok(EVERYTHING.text):>8}  "
      f"{EVERYTHING.manifest['surface_digest']}")
print("\nartifact versions in this run:")
for a in surface:
    print(f"   {a.kind:9} {a.name:22} {a.version:8} {a.digest}")


tone      audience      words  tokens  surface digest
formal    customer        585     796  bafe0a0338aa
formal    developer       620     860  8e303b946179
casual    customer        584     788  61368dc1fa99
casual    developer       602     829  d43c2bf1becd

everything, all variants    837    1177  30ef3f020538

artifact versions in this run:
   identity  acme-docs-agent        2.1.0    f13e99c92930
   project   acme-data-platform     4.0.2    6246496911d7
   skill     release-notes          3.4.0    9a0b8bce844f
   persona   plainspoken            1.2.0    dc78ba66e414


## 10. What this pass cost


In [ ]:
@lru_cache(maxsize=1)
def pricing() -> Dict[str, Tuple[float, float]]:
    """(prompt, completion) USD per token, from OpenRouter's own catalogue."""
    req = urllib.request.Request(
        f"{BASE_URL}/models",
        headers={"Authorization": f"Bearer {os.environ['OPENROUTER_API_KEY']}"})
    data = json.load(urllib.request.urlopen(req, timeout=60))["data"]
    return {m["id"]: (float(m["pricing"]["prompt"]), float(m["pricing"]["completion"]))
            for m in data}


def cost_of(u) -> float:
    p_in, p_out = pricing().get(u["model"], (0.0, 0.0))
    return u["prompt_tokens"] * p_in + u["completion_tokens"] * p_out


print(f"{'call':<22}{'model':<24}{'tok in':>8}{'tok out':>9}{'sec':>7}{'USD':>9}")
for u in USAGE:
    print(f"{u['label']:<22}{u['model']:<24}{u['prompt_tokens']:>8}"
          f"{u['completion_tokens']:>9}{u['seconds']:>7.1f}{cost_of(u):>9.5f}")
print(f"{'total':<22}{'':<24}{sum(u['prompt_tokens'] for u in USAGE):>8}"
      f"{sum(u['completion_tokens'] for u in USAGE):>9}"
      f"{sum(u['seconds'] for u in USAGE):>7.1f}{sum(cost_of(u) for u in USAGE):>9.5f}")


call                  model                     tok in  tok out    sec      USD
select[unambiguous/as written]openai/gpt-5.4-mini          209       39    1.2  0.00033
select[unambiguous/as written]openai/gpt-5.4-mini          209       38    0.9  0.00033
select[unambiguous/as written]openai/gpt-5.4-mini          209       36    0.8  0.00032
select[unambiguous/degraded]openai/gpt-5.4-mini          185       35    1.0  0.00030
select[unambiguous/degraded]openai/gpt-5.4-mini          185       34    0.8  0.00029
select[unambiguous/degraded]openai/gpt-5.4-mini          185       34    0.9  0.00029
select[ambiguous/as written]openai/gpt-5.4-mini          213       40    1.1  0.00034
select[ambiguous/as written]openai/gpt-5.4-mini          213       37    0.9  0.00033
select[ambiguous/as written]openai/gpt-5.4-mini          213       37    0.9  0.00033
select[ambiguous/degraded]openai/gpt-5.4-mini          189       37    0.9  0.00031
select[ambiguous/degraded]openai/gpt-5.4-mini          1

The estimates use published per-token rates and the measured tokens. Provider accounting may round or settle later, so these numbers describe the approximate cost of this run.


### Adapting this to your own agent

1. **Write the identity first**, and put in it only what is true for every task. If a rule needs a qualifier it belongs in a skill.
2. **Give every rule a section id** before you write any machinery Everything in this notebook, and all of chapter 4, is built on rules being addressable.
3. **Write `when_to_use` as a boundary**, including what the skill is not for. Test it with a table of tasks and expected selections.
4. **Decide who owns each fact.** A fact the harness knows (`breaking`) is
   supplied, never inferred — it costs no inference and cannot be hallucinated.
5. **Put the surface digest in your traces** from day one. Retrofitting it means re-running everything you wanted to explain.
